# Dementia (3): Higher-order syntax (entropy rate and sample entropy)

**Aim:**
Compare control (CN) vs. Alzheimer disease (AD) subjects using the metrics
**entropy rate (ER)** and **sample entropy (SE)**, across a range of history lengths.

This notebook reproduces the results published in:
_von Wegner, F.; Hermann, G.; Tödt, I.; Todtenhaupt, I. K. & Laufs, H.
A quantitative comparison of two methods for higher-order EEG microstate syntax analysis 
Brain Topography, 2026_ 

**Data**: Dementia EEG microstate sequences (Miltiadous et al.)

**Note:** We have excluded the third group (fronto-temporal dementia) from this analysis.

In [ ]:
# JupyterLite/Pyodide only: install the pure-Python mstsa build (no numba/C
# extensions -- see https://github.com/Frederic-vW/mstsa/tree/pyodide). Falls
# through silently on a normal Jupyter install, where mstsa is already present.
try:
    import piplite
    await piplite.install(
        "https://raw.githubusercontent.com/Frederic-vW/mstsa/pyodide/wheels/mstsa-0.4.3-py3-none-any.whl"
    )
except ImportError:
    pass


In [ ]:
import os

In [ ]:
data_path = "data/CN_AD"  # only participants.tsv is bundled in this copy;
                           # per-subject caches are looked up by filename only

## Collate a subject list
- Same criteria and data as used in [A Quantitative Comparison of Two Methods for Higher-Order EEG
Microstate Syntax Analysis](https://doi.org/10.1007/s10548-026-01196-5)
- As the methods used below are data hungry, we only use sequences longer than 5 
minutes, i.e. 150,000 samples (500 Hz sampling rate), which reduces the number of 
subjects compared to the full public database.
- Across all included subjects, the shortest recording has 153303 samples (306.6 s), 
the longest 442033 samples (884.066 s).

In [ ]:
# subject/group mapping. participants.tsv lists the full original dataset
# (all 3 groups); only a curated subset actually has data here. That subset
# is exactly what's been cached (data/cache/mc_surrogates/), which this copy
# bundles instead of the raw MPILMBB/CN_AD .npy files -- so derive the real
# subject list from the cache directory itself, then split by group via
# participants.tsv.
f_participants = f"{data_path}/participants.tsv"
with open(f_participants, 'r') as fp:
    lines = fp.readlines()
header = lines[0].strip().split('\t')
lines = [l.strip().split('\t') for l in lines[1:]]
group_of = {l[0]: l[3] for l in lines}

cached_subjects = sorted(f.split('_')[0] for f in os.listdir("data/cache/mc_surrogates")
                          if f.endswith('.npz'))
subjects_CN = [s for s in cached_subjects if group_of.get(s) == 'C']
subjects_AD = [s for s in cached_subjects if group_of.get(s) == 'A']
subjects = subjects_CN + subjects_AD

# microstate sequence filenames
ms_files = {
    'CN': sorted([f"{data_path}/{subj}_EC_ms_K4.npy" for subj in subjects_CN]),
    'AD': sorted([f"{data_path}/{subj}_EC_ms_K4.npy" for subj in subjects_AD]),
}
print(f"Control (CN): n = {len(ms_files['CN'])} subjects")
print(f"Alzheimer Disease (AD): n = {len(ms_files['AD'])} subjects")


## Theoretical entropy rate (ER) and sample entropy (SE) values for first-order syntax

- As the aim is to implement and evaluate metrics of higher-order syntax, we will 
first establish the expected baseline values for first-order Markov chains. Anything 
deviating from a first-order structure will be classified as higher-order syntax.
- The source article derives estimates for entropy rate (ER) and sample entropy (SE). 
Their numerical verification is again included here.
- For each subject's **continuous microstate sequence**, truncated to `n_max=100000` 
samples, we:
    - estimate the empirical transition matrix `T` and stationary distribution `p`,
    - compute theoretical ER/SE for the Markov-0 (i.i.d.) and Markov-1 models from `T`/`p`
  (`entropy_rates_mc`, `sample_entropies_mc`
    - generate Markov-0 and Markov-1 surrogate sequences matched in length to the real data
  (`mc_sample_path`), plus a Markov-1 *jump-chain* surrogate of exactly `n_max` samples
    - extract the jump (embedded) process from the continuous surrogates (`embedded_process`),
    - compute empirical ER (`entropy_rate`, `kmax=k` for `k=1..6`) and SE (`sample_entropy_disc`,
  template lengths `m=1..6`) on the surrogates,
  - average across subjects in a group (CN / AD) to get mean +/- std at each `k`.
- Per-subject results are cached to disk (`data/cache/mc_surrogates/*.npz`), so 
re-running only computes subjects that aren't cached yet
- `entropy_rate` and `sample_entropy_disc` are C-accelerated functions

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
import mstsa

K = 4
n_max = int(1e5)
kmax = 6
se_M = 6
n_long_factor = 10

cache_dir = "data/cache/mc_surrogates"
os.makedirs(cache_dir, exist_ok=True)

In [ ]:
_K_KEYS = ['er_cont_mc0_k', 'er_cont_mc1_k', 'er_jump_mc1_k', 'er_jump_mc1_long_k',
           'se_cont_mc0_k', 'se_cont_mc1_k', 'se_jump_mc1_k',
           'er_real_k', 'se_real_k', 'er_jump_real_k', 'se_jump_real_k']
_THEORY_KEYS = ['er_cont_mc0', 'er_cont_mc1', 'er_jump_mc1',
                'se_cont_mc0', 'se_cont_mc1', 'se_jump_mc1']


def _compute_subject(f, K, n_max, kmax, se_M, n_long_factor):
    x = np.load(f).astype(np.int32)
    x = x[:n_max]
    n = len(x)

    T = mstsa.tpm_cond(x, K)
    p = mstsa.p_stationary(T)

    # theoretical ER/SE, Markov-0 and Markov-1 (valid for all k)
    er_cont_mc0, _, er_cont_mc1, er_jump_mc1 = mstsa.entropy_rates_mc(x, K)
    se_cont_mc0, _, se_cont_mc1, se_jump_mc1 = mstsa.sample_entropies_mc(x, K)

    # Markov-0 (iid) and Markov-1 surrogates, matched in length to x
    T0 = np.tile(p, (K, 1))
    surr_mc0 = mstsa.mc_sample_path(T=T0, p=p, n=n)
    surr_mc1 = mstsa.mc_sample_path(T=T, p=p, n=n)
    jump_mc1, _ = mstsa.embedded_process(surr_mc1)

    # embedded/jump chain's own transition matrix: remove self-transitions from T and
    # re-normalize rows (same construction as mstsa.generator_matrix's T_jump). Sampling
    # directly from this matrix never self-transitions, so n=n_max gives exactly n_max
    # jump samples in one shot -- no need to oversample a continuous surrogate and trim.
    T_diag = np.diag(T)
    T_jump = T - np.diag(T_diag)
    T_jump /= T_jump.sum(axis=1, keepdims=True)
    p_jump = mstsa.p_stationary(T_jump)
    jump_mc1_long = mstsa.mc_sample_path(T=T_jump, p=p_jump, n=n_max)

    # jump (embedded) process of the real EEG sequence itself
    jump_real, _ = mstsa.embedded_process(x)

    # ER: kmax=k for k=1..kmax (history length k). SE: sample_entropy_disc(m=se_M) returns
    # template lengths order=0..se_M in one call; we keep order=1..se_M (order=0 is the
    # marginal collision entropy, not comparable to the theoretical pair-based formulas).
    ks = range(1, kmax + 1)
    er_cont_mc0_k = np.array([mstsa.entropy_rate(surr_mc0, K, kmax=k)[0] for k in ks])
    er_cont_mc1_k = np.array([mstsa.entropy_rate(surr_mc1, K, kmax=k)[0] for k in ks])
    er_jump_mc1_k = np.array([mstsa.entropy_rate(jump_mc1, K, kmax=k)[0] for k in ks])
    er_jump_mc1_long_k = np.array([mstsa.entropy_rate(jump_mc1_long, K, kmax=k)[0] for k in ks])
    er_real_k = np.array([mstsa.entropy_rate(x, K, kmax=k)[0] for k in ks])
    er_jump_real_k = np.array([mstsa.entropy_rate(jump_real, K, kmax=k)[0] for k in ks])

    se_cont_mc0_k = mstsa.sample_entropy_disc(surr_mc0, m=se_M)[1:]
    se_cont_mc1_k = mstsa.sample_entropy_disc(surr_mc1, m=se_M)[1:]
    se_jump_mc1_k = mstsa.sample_entropy_disc(jump_mc1, m=se_M)[1:]
    se_real_k = mstsa.sample_entropy_disc(x, m=se_M)[1:]
    se_jump_real_k = mstsa.sample_entropy_disc(jump_real, m=se_M)[1:]

    return dict(
        er_cont_mc0_k=er_cont_mc0_k, er_cont_mc1_k=er_cont_mc1_k,
        er_jump_mc1_k=er_jump_mc1_k, er_jump_mc1_long_k=er_jump_mc1_long_k,
        se_cont_mc0_k=se_cont_mc0_k, se_cont_mc1_k=se_cont_mc1_k, se_jump_mc1_k=se_jump_mc1_k,
        er_real_k=er_real_k, se_real_k=se_real_k,
        er_jump_real_k=er_jump_real_k, se_jump_real_k=se_jump_real_k,
        er_cont_mc0=er_cont_mc0, er_cont_mc1=er_cont_mc1, er_jump_mc1=er_jump_mc1,
        se_cont_mc0=se_cont_mc0, se_cont_mc1=se_cont_mc1, se_jump_mc1=se_jump_mc1,
    )


def analyze_subject(f, K=K, n_max=n_max, kmax=kmax, se_M=se_M,
                     n_long_factor=n_long_factor, cache_dir=cache_dir, force=False):
    cache_file = os.path.join(cache_dir, os.path.basename(f).replace('.npy', '_mcstats.npz'))
    if os.path.exists(cache_file) and not force:
        d = np.load(cache_file)
        if all(k in d.files for k in _K_KEYS):
            return {k: d[k] for k in d.files}
    res = _compute_subject(f, K, n_max, kmax, se_M, n_long_factor)
    np.savez(cache_file, **res)
    return res


def analyze_group(files, max_subjects=None, **kwargs):
    files = files[:max_subjects] if max_subjects else files
    results = []
    for i, f in enumerate(files):
        cache_file = os.path.join(cache_dir, os.path.basename(f).replace('.npy', '_mcstats.npz'))
        cached = os.path.exists(cache_file)
        t0 = time.time()
        res = analyze_subject(f, **kwargs)
        tag = "cached" if cached else f"{time.time() - t0:.1f}s"
        print(f"  [{i + 1}/{len(files)}] {os.path.basename(f)} ({tag})")
        results.append(res)
    return results


def aggregate_group(results):
    stats = {}
    for k in _K_KEYS:
        arr = np.array([r[k] for r in results])
        stats[k + '_mean'] = arr.mean(axis=0)
        stats[k + '_std'] = arr.std(axis=0)
    for k in _THEORY_KEYS:
        arr = np.array([r[k] for r in results])
        stats[k + '_theory_mean'] = arr.mean()
    return stats

In [ ]:
max_subjects = None  # full population

group_results = {}
group_stats = {}
for group in ['CN', 'AD']:
    print(f"=== {group} ===")
    group_results[group] = analyze_group(ms_files[group], max_subjects=max_subjects)
    group_stats[group] = aggregate_group(group_results[group])

In [ ]:
YLIM_CONTINUOUS = (0.20, 2.25)
YLIM_JUMP = (1.3, 1.6)


def plot_group(stats, group, kmax=kmax):
    ks = np.arange(1, kmax + 1)
    fig, axes = plt.subplots(2, 2, figsize=(12, 9))

    # continuous sequences, entropy rate
    ax = axes[0, 0]
    ax.axhline(stats['er_cont_mc0_theory_mean'], color='r', ls=':', label='ER theoretical MC-0')
    ax.axhline(stats['er_cont_mc1_theory_mean'], color='b', ls=':', label='ER theoretical MC-1')
    ax.errorbar(ks, stats['er_cont_mc0_k_mean'], yerr=stats['er_cont_mc0_k_std'],
                fmt='o', mfc='none', color='r', capsize=3, label='ER surrogate MC-0')
    ax.errorbar(ks, stats['er_cont_mc1_k_mean'], yerr=stats['er_cont_mc1_k_std'],
                fmt='o', mfc='none', color='b', capsize=3, label='ER surrogate MC-1')
    ax.set_xlabel('k')
    ax.set_ylabel('ER (bits/sample)')
    ax.set_ylim(YLIM_CONTINUOUS)
    ax.set_title(f'{group}: continuous sequences, entropy rate')
    ax.legend()

    # continuous sequences, sample entropy
    ax = axes[0, 1]
    ax.axhline(stats['se_cont_mc0_theory_mean'], color='r', ls=':', label='SE theoretical MC-0')
    ax.axhline(stats['se_cont_mc1_theory_mean'], color='b', ls=':', label='SE theoretical MC-1')
    ax.errorbar(ks, stats['se_cont_mc0_k_mean'], yerr=stats['se_cont_mc0_k_std'],
                fmt='o', mfc='none', color='r', capsize=3, label='SE surrogate MC-0')
    ax.errorbar(ks, stats['se_cont_mc1_k_mean'], yerr=stats['se_cont_mc1_k_std'],
                fmt='o', mfc='none', color='b', capsize=3, label='SE surrogate MC-1')
    ax.set_xlabel('k')
    ax.set_ylabel('SE (bits)')
    ax.set_ylim(YLIM_CONTINUOUS)
    ax.set_title(f'{group}: continuous sequences, sample entropy')
    ax.legend()

    # jump sequences, entropy rate
    ax = axes[1, 0]
    ax.axhline(stats['er_jump_mc1_theory_mean'], color='b', ls=':', label='ER theoretical MC-1')
    ax.errorbar(ks, stats['er_jump_mc1_k_mean'], yerr=stats['er_jump_mc1_k_std'],
                fmt='o', mfc='none', color='b', capsize=3, label='ER surrogate MC-1')
    ax.errorbar(ks, stats['er_jump_mc1_long_k_mean'], yerr=stats['er_jump_mc1_long_k_std'],
                fmt='s', mfc='none', color='g', capsize=3, label='ER long surrogate MC-1 (n=100000)')
    ax.set_xlabel('k')
    ax.set_ylabel('ER (bits/sample)')
    ax.set_ylim(YLIM_JUMP)
    ax.set_title(f'{group}: jump sequences, entropy rate')
    ax.legend()

    # jump sequences, sample entropy
    ax = axes[1, 1]
    ax.axhline(stats['se_jump_mc1_theory_mean'], color='b', ls=':', label='SE theoretical MC-1')
    ax.errorbar(ks, stats['se_jump_mc1_k_mean'], yerr=stats['se_jump_mc1_k_std'],
                fmt='o', mfc='none', color='b', capsize=3, label='SE surrogate MC-1')
    ax.set_xlabel('k')
    ax.set_ylabel('SE (bits)')
    ax.set_ylim(YLIM_JUMP)
    ax.set_title(f'{group}: jump sequences, sample entropy')
    ax.legend()

    plt.tight_layout()
    os.makedirs('figures', exist_ok=True)
    plt.savefig(f'figures/03_group_{group}.png', dpi=150, bbox_inches='tight')
    plt.show()

for group in ['CN']: # ['CN', 'AD']
    plot_group(group_stats[group], group)

**Observations:**
1. Good agreement between theoretical and numerical ER and SE values. The approximation formulas are probably not bad.
2. Bias: there is a clear downward bias in ER estimates on jump sequences for larger history lengths `k` (lower left panels). This is caused by the relatively short sequence length and fixed by increasing the surrogate lengths (green squares). It can, however, be an issue in real data that cannot be lengthened artificially. 

## Real EEG sequences vs. first-order Markov (MC-1) surrogates

The next question is how ER and SE for microstate sequences, both continuous and jump, behave relative to the sequence's own first-order syntax level. 

The figure below creates the first-order Markov (MC-1) ER and SE levels for different history lengths (=block size = microstate word legnth) $k$ and shows them in blue.
Empirical microstate sequence ER and SE curves are shown in black.

In [ ]:
def half_errorbar(ax, ks, mean_a, std_a, label_a, mean_b, std_b, label_b,
                   color_a='k', color_b='b', fmt_a='o-', fmt_b='o-', a_is_lower=None):
    """Plot two close curves with half error bars: whichever curve is lower gets only
    its downward whisker, the higher one only its upward whisker, applied with full
    magnitude across all k so the color/direction pairing never flips at k=1 even
    though the two means can be nearly equal there. Pass `a_is_lower` explicitly to
    fix the assignment (recommended when the two curves are close enough that an
    automatic per-dataset decision could disagree between panels); leave as None to
    auto-decide from the median of (mean_a - mean_b) over k=2..end."""
    if a_is_lower is None:
        a_is_lower = np.median((mean_a - mean_b)[1:]) <= 0 if len(mean_a) > 1 else mean_a[0] <= mean_b[0]
    err_a = np.zeros((2, len(mean_a)))
    err_b = np.zeros((2, len(mean_b)))
    if a_is_lower:
        err_a[0] = std_a
        err_b[1] = std_b
    else:
        err_a[1] = std_a
        err_b[0] = std_b
    ax.errorbar(ks, mean_a, yerr=err_a, fmt=fmt_a, mfc='none', color=color_a, capsize=3, label=label_a)
    ax.errorbar(ks, mean_b, yerr=err_b, fmt=fmt_b, mfc='none', color=color_b, capsize=3, label=label_b)


# Fixed half-error-bar orientation per panel (same for CN and AD): real EEG is the
# lower curve (downward whisker) everywhere except continuous sample entropy, where
# it's the upper curve. Set explicitly rather than auto-detected per group/panel.
_A_IS_LOWER_REAL = {
    'er_cont': True,   # continuous ER: real down, surrogate up
    'se_cont': False,  # continuous SE: real up, surrogate down
    'er_jump': True,   # jump ER: real down, surrogate up
    'se_jump': True,   # jump SE: real down, surrogate up
}


def plot_real_vs_mc1(stats, group, kmax=kmax):
    ks = np.arange(1, kmax + 1)
    fig, axes = plt.subplots(2, 2, figsize=(12, 9))

    # continuous sequences, entropy rate
    ax = axes[0, 0]
    half_errorbar(ax, ks, stats['er_real_k_mean'], stats['er_real_k_std'], 'ER real EEG',
                  stats['er_cont_mc1_k_mean'], stats['er_cont_mc1_k_std'], 'ER surrogate MC-1',
                  a_is_lower=_A_IS_LOWER_REAL['er_cont'])
    ax.set_xlabel('k')
    ax.set_ylabel('ER (bits/sample)')
    ax.set_title(f'{group}: continuous sequences, entropy rate')
    ax.legend()

    # continuous sequences, sample entropy
    ax = axes[0, 1]
    half_errorbar(ax, ks, stats['se_real_k_mean'], stats['se_real_k_std'], 'SE real EEG',
                  stats['se_cont_mc1_k_mean'], stats['se_cont_mc1_k_std'], 'SE surrogate MC-1',
                  a_is_lower=_A_IS_LOWER_REAL['se_cont'])
    ax.set_xlabel('k')
    ax.set_ylabel('SE (bits)')
    ax.set_title(f'{group}: continuous sequences, sample entropy')
    ax.legend()

    # jump sequences, entropy rate
    ax = axes[1, 0]
    half_errorbar(ax, ks, stats['er_jump_real_k_mean'], stats['er_jump_real_k_std'], 'ER jump real EEG',
                  stats['er_jump_mc1_k_mean'], stats['er_jump_mc1_k_std'], 'ER surrogate MC-1',
                  a_is_lower=_A_IS_LOWER_REAL['er_jump'])
    ax.set_xlabel('k')
    ax.set_ylabel('ER (bits/sample)')
    ax.set_title(f'{group}: jump sequences, entropy rate')
    ax.legend()

    # jump sequences, sample entropy
    ax = axes[1, 1]
    half_errorbar(ax, ks, stats['se_jump_real_k_mean'], stats['se_jump_real_k_std'], 'SE jump real EEG',
                  stats['se_jump_mc1_k_mean'], stats['se_jump_mc1_k_std'], 'SE surrogate MC-1',
                  a_is_lower=_A_IS_LOWER_REAL['se_jump'])
    ax.set_xlabel('k')
    ax.set_ylabel('SE (bits)')
    ax.set_title(f'{group}: jump sequences, sample entropy')
    ax.legend()

    plt.tight_layout()
    os.makedirs('figures', exist_ok=True)
    plt.savefig(f'figures/03_real_vs_mc1_{group}.png', dpi=150, bbox_inches='tight')
    plt.show()


for group in ['CN', 'AD']:
    plot_real_vs_mc1(group_stats[group], group)

**Observations:**
1. Continuous sequences / ER: real < MC-1; real data is more regular and predictable
2. Continuous sequences / SE: real > MC-1; real data is less regular and predictable
3. Jump sequences / ER: real < MC-1; real data is more regular and predictable
4. Jump sequences / SE: real < MC-1; real data is more regular and predictable
5. Observations 1-4 hold for control (CN) and dementia (AD) groups
6. Entropy rate and sample entropy identify different types of randomness: ER analyzes a short tail of symbols preceding the current label whereas SE scans the _whole_ sequence for word recurrences and is therefore more strongly affected by slow drifts or modulations in syntax across the 5 minute recording
7. Jump sequence / ER suffer from small-N downward bias (MC-1 should be flat)

## CN vs. AD: real EEG sequences, no surrogates

- The next cell compares ER and SE between the groups (CN, AD) directly
- Can these metrics distinguish the groups? Potential biomarker?

In [ ]:
def plot_cn_vs_ad(cn_stats, ad_stats, kmax=kmax):
    ks = np.arange(1, kmax + 1)
    fig, axes = plt.subplots(2, 2, figsize=(12, 9))

    # continuous sequences, entropy rate
    ax = axes[0, 0]
    half_errorbar(ax, ks, cn_stats['er_real_k_mean'], cn_stats['er_real_k_std'], 'ER CN',
                  ad_stats['er_real_k_mean'], ad_stats['er_real_k_std'], 'ER AD',
                  color_a='tab:blue', color_b='tab:red')
    ax.set_xlabel('k')
    ax.set_ylabel('ER (bits/sample)')
    ax.set_title('continuous sequences, entropy rate')
    ax.legend()

    # continuous sequences, sample entropy
    ax = axes[0, 1]
    half_errorbar(ax, ks, cn_stats['se_real_k_mean'], cn_stats['se_real_k_std'], 'SE CN',
                  ad_stats['se_real_k_mean'], ad_stats['se_real_k_std'], 'SE AD',
                  color_a='tab:blue', color_b='tab:red')
    ax.set_xlabel('k')
    ax.set_ylabel('SE (bits)')
    ax.set_title('continuous sequences, sample entropy')
    ax.legend()

    # jump sequences, entropy rate
    ax = axes[1, 0]
    half_errorbar(ax, ks, cn_stats['er_jump_real_k_mean'], cn_stats['er_jump_real_k_std'], 'ER jump CN',
                  ad_stats['er_jump_real_k_mean'], ad_stats['er_jump_real_k_std'], 'ER jump AD',
                  color_a='tab:blue', color_b='tab:red')
    ax.set_xlabel('k')
    ax.set_ylabel('ER (bits/sample)')
    ax.set_title('jump sequences, entropy rate')
    ax.legend()

    # jump sequences, sample entropy
    ax = axes[1, 1]
    half_errorbar(ax, ks, cn_stats['se_jump_real_k_mean'], cn_stats['se_jump_real_k_std'], 'SE jump CN',
                  ad_stats['se_jump_real_k_mean'], ad_stats['se_jump_real_k_std'], 'SE jump AD',
                  color_a='tab:blue', color_b='tab:red')
    ax.set_xlabel('k')
    ax.set_ylabel('SE (bits)')
    ax.set_title('jump sequences, sample entropy')
    ax.legend()

    plt.tight_layout()
    os.makedirs('figures', exist_ok=True)
    plt.savefig('figures/03_cn_vs_ad.png', dpi=150, bbox_inches='tight')
    plt.show()


plot_cn_vs_ad(group_stats['CN'], group_stats['AD'])

**Conclusions:**
1. Overall, a complex picture emerges: entropy rate (ER) and sample entropy (SE) 
reflect different aspects of microstate dynamics
2. Continuous sequences: CN > AD (ER and SE); control subject sequences are less predictable, more surprising, than AD subjects
3. Jump sequences: AD > CN (ER and SE), similar to `08_dementia_complexity.ipynb`. 
Microtate sequences from AD subjects are less predictable.
4. **Interpretation**: AD causes generalized EEG slowing. This is probably reflected in continuous sequences, making them evolve slower and thus more predictable State-to-state predictability decreases (less regular, more chaotic'). If one accepts the microstate-functional brain networks association, this could be interpreted as less structured brain state activations.
5. This interpretation cannot be derived from classical microstate parameters, exemplifying the use of higher-order methods.